<a href="https://colab.research.google.com/github/vthkrl/big_data_fall26/blob/main/Big_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



1. Data consolidation: combine information from a wide range of data sources to build a comprehensive picture of each NYC community district.




Notes on datasets:
- Facility info is in `facilities.csv`, the `cd` column lists the district number
- Data on crime rate is in `neighborhood_indicators.xlsx`, and needs to be extracted (`crime_all_rt` column)
- Median income is in `median-income.csv` with the *name* of the CD in the `location_name` column (name can be mapped to district number using the neighborhood indicators table)
- Remaining metrics each have their own csv file, each of which has district number in the `GeoID` column
- Some metrics are grouped by individual year while others are in four-year blocks. To account for this, we could potentially take the single-year data and average every block of four years to create corresponding block values.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/Project/datasets/"

spark = SparkSession.builder \
    .appName("FacilityPredictor") \
    .getOrCreate()

facilities_path = folder_path + "facilities.csv"
facilities = spark.read.csv(facilities_path, header=True, inferSchema=True)

neighborhood_indicators_path = folder_path + "neighborhood_indicators.xlsx"
neighborhood_indicators = spark.read.csv(neighborhood_indicators_path, header=True, inferSchema=True)

# EH data
high_school_graduation_rate_path = folder_path + "high_school.csv"
high_school_graduation_rate = spark.read.csv(high_school_graduation_rate_path, header=True, inferSchema=True)

premature_mortality_rate_path = folder_path + "mortality.csv"
premature_mortality_rate = spark.read.csv(premature_mortality_rate_path, header=True, inferSchema=True)

2. Data cleaning: standardize date ranges and spatial units from each data source, handle facilities with poor or absent location information.


In [ ]:
# community districts
cds = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 501, 502, 503]
cds_tuples = [(cd,) for cd in cds]
scores = spark.createDataFrame(cds_tuples, schema=["cd"])

# clean facilities
grouped_facilities = facilities.groupBy("cd").agg(F.collect_list(F.col("facgroup"))).alias("Facilities")

scores = scores.join(
    grouped_facilities,
    on="cd",
    how="left"
)

# clean neighborhood_indicators

# clean health data
scores = scores.join(
    high_school_graduation_rate.select(
        high_school_graduation_rate["GeoID"].alias("cd"),
        F.col("Percent").alias("High school graduation rate"),
    ),
    on="cd",
    how="left"
)
scores = scores.join(
    premature_mortality_rate.select(premature_mortality_rate["GeoID"].alias("cd"), "Age-adjusted rate per 100,000"),
    on="cd",
    how="left"
)

scores.show()

3. Model training: train a machine learning model on the raw scores for the quality of life domains of each district, so that the model determines how the count of a facility type in a district may affect the raw scores of domains for that district.


4. Model execution: for each (district, facility type) pair, use the model to calculate the predicted domain scores, then calculate the predicted domain percentiles and the Predicted Overall Percentile Lift.


5. Civil planning: for each district, compare the Predicted Overall Percentile Lift of every facility type to determine which facility type would most improve the quality of life for that district, and for the city as a whole, compare the highest Predicted Overall Percentile Lift from each district to determine which facility types should be built and in which districts they should be built.


6. Interactive output: display both the input data and our calculated results on a user-friendly interactive map to aid in interpretation.